# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [2]:
from _imports import * # Centralized file containing all imports

2025-05-02 09:22:36.349160: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-02 09:22:36.361072: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746188556.374460   26551 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746188556.377946   26551 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-02 09:22:36.391008: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [3]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
troo.get_gpu_info()

TensorFlow Version: 2.18.0
CUDA support detected
  CUDA Version: 12.5.1
  cuDNN Version: 9

GPUs Detected (1): ['/physical_device:GPU:0']
Default GPU device: /device:GPU:0


2025-05-02 09:22:37.899069: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1746188557.899114   26551 gpu_process_state.cc:201] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1746188557.899354   26551 gpu_device.cc:2022] Created device /device:GPU:0 with 2229 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


## 2. Run Parameters 

In [4]:
NUM_TRIALS = 10
EPOCHS = 1
TOP_K = 1  # Number of top trials to save

mixed_precision.set_global_policy("mixed_float16")

#? Set to an existing path to resume training
RESUME_TRAINING_PATH = None # None or "runs/nas_1" 

In [5]:
RUN_DIR = RESUME_TRAINING_PATH or troo.create_run_directory(prefix="nas_")
print(f"Run directory: {RUN_DIR}")

Run directory: runs/nas_1


## 3. Data Loading and Preprocessing

In [6]:
def convert_to_sparse_labels(y: np.ndarray) -> np.ndarray:
    """
    Converts beam score targets to sparse integer labels suitable for SparseCategoricalCrossentropy loss.

    Args:
        y (np.ndarray): Original beam scores with shape (N, 8, 32).

    Returns:
        np.ndarray: Array of integer labels with shape (N,), where each label corresponds 
                    to the index (flattened over 8x32) of the maximum score.
    """
    # Reshape the input so that each sample becomes a 1D array (e.g., 256 elements)
    y_flat = y.reshape(y.shape[0], -1)
    # For each sample, return the index of the maximum value
    labels = np.argmax(y_flat, axis=1)
    return labels


In [7]:
# Define the base directory for data files
DATA_DIR = "./data/s008"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s008_y_train = np.load(beam_output_path)
s008_coord_input = np.load(coord_input_path)
s008_image_input = np.load(image_input_path)
s008_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s008_y_train = s008_y_train.astype(np.float32)
s008_coord_input = s008_coord_input.astype(np.float32)

print(f"Shape before conversion: {s008_y_train.shape}")
s008_y_train = convert_to_sparse_labels(s008_y_train)
print(f"Shape after conversion: {s008_y_train.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s008_y_train.shape}")
print(f"coord_input shape: {s008_coord_input.shape}")
print(f"image_input shape: {s008_image_input.shape}")
print(f"lidar_input shape: {s008_lidar_input.shape}")

Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_train shape: (1960,)
coord_input shape: (1960, 2)
image_input shape: (1960, 48, 81, 1)
lidar_input shape: (1960, 20, 200, 10)


/tmp/ipykernel_26551/1378391195.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)


In [8]:
# Define the base directory for data files
DATA_DIR = "./data/s009"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s009_y = np.load(beam_output_path)
s009_coord_input = np.load(coord_input_path)
s009_image_input = np.load(image_input_path)
s009_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s009_y = s009_y.astype(np.float32)
s009_coord_input = s009_coord_input.astype(np.float32)

print(f"Shape before conversion: {s009_y.shape}")
s009_y = convert_to_sparse_labels(s009_y)
print(f"Shape after conversion: {s009_y.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s009_y.shape}")
print(f"coord_input shape: {s009_coord_input.shape}")
print(f"image_input shape: {s009_image_input.shape}")
print(f"lidar_input shape: {s009_lidar_input.shape}")

Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y_train shape: (9638,)
coord_input shape: (9638, 2)
image_input shape: (9638, 48, 81, 1)
lidar_input shape: (9638, 20, 200, 10)


/tmp/ipykernel_26551/428444760.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [9]:
# # ----------------------- Subsample dataset for testing ---------------------- #
# SAMPLE_SIZE = 50  # Use a subset of x samples for testing
# s008_y_train = s008_y_train[:SAMPLE_SIZE]
# s008_coord_input = s008_coord_input[:SAMPLE_SIZE]
# s008_image_input = s008_image_input[:SAMPLE_SIZE]
# s008_lidar_input = s008_lidar_input[:SAMPLE_SIZE]

# # Print the shapes of the loaded data
# print(f"y_train s008 shape: {s008_y_train.shape}")
# print(f"coord_input s008 shape: {s008_coord_input.shape}")
# print(f"image_input s008 shape: {s008_image_input.shape}")
# print(f"lidar_input s008 shape: {s008_lidar_input.shape}")

# s009_y_train = s009_y_train[:SAMPLE_SIZE]
# s009_coord_input = s009_coord_input[:SAMPLE_SIZE]
# s009_image_input = s009_image_input[:SAMPLE_SIZE]
# s009_lidar_input = s009_lidar_input[:SAMPLE_SIZE]

# print(f"y_train s009 shape: {s009_y_train.shape}")
# print(f"coord_input s009 shape: {s009_coord_input.shape}")
# print(f"image_input s009 shape: {s009_image_input.shape}")
# print(f"lidar_input s009 shape: {s009_lidar_input.shape}")

## 4. Getters

### 4.1. Regularizers

In [10]:
def get_regularizer(trial: optuna.Trial, name: str) -> Optional[tf.keras.regularizers.Regularizer]:
    """
    Suggests a regularization strategy using Optuna and returns the corresponding Keras regularizer.
    
    Args:
        trial (optuna.Trial): Optuna trial object used to sample the regularizer.
        name (str): Unique identifier for this regularizer parameter (used as key).

    Returns:
        Optional[tf.keras.regularizers.Regularizer]: The selected Keras regularizer instance,
        or `None` if "none" was selected.
    """
    # Suggest a regularizer type
    reg_type: str = trial.suggest_categorical(
        name,
        [
            "none",
            "l1",
            "l2",
            "l1l2",
            # "orthogonal",  #! only works for rank-2 tensors
        ],
    )

    # Map each regularizer name to a corresponding Keras regularizer instance
    regularizer_map: Dict[str, Optional[tf.keras.regularizers.Regularizer]] = {
        "none": None,
        "l1": regularizers.L1(l1=0.01),
        "l2": regularizers.L2(l2=0.01),
        "l1l2": regularizers.L1L2(l1=0.01, l2=0.01),
        "orthogonal": regularizers.OrthogonalRegularizer(factor=0.01, mode="rows"),
    }

    # Return the appropriate regularizer, or None if not found
    return regularizer_map.get(reg_type, None)

### 4.2. Activation Functions

In [11]:
def get_activation(trial: Any, name: str) -> Union[str, Callable[..., layers.Layer]]:
    """
    Suggests an activation function from a predefined list using Optuna.

    Args:
        trial (Any): The Optuna trial instance used to suggest a value.
        name (str): A unique name for this hyperparameter (e.g., "layer_1_activation").

    Returns:
        Union[str, Callable[..., layers.Layer]]: A string representing the activation function.
        This can be passed directly into a Keras layer's `activation=` argument.
    """
    return trial.suggest_categorical(
        name,
        [
            "relu",
            "tanh",
            "sigmoid",  # Logistic
            "elu", 
            "swish",  # x * sigmoid(x)
            "leaky_relu",
        ],
    )

### 4.3. Optimizers

In [12]:
def get_optimizer(trial: optuna.Trial) -> tf.keras.optimizers.Optimizer:
    """
    Suggests and returns a TensorFlow optimizer with a trial-based learning rate.

    Args:
        trial (optuna.Trial): Optuna trial object used for hyperparameter suggestion.

    Returns:
        tf.keras.optimizers.Optimizer: An instance of the selected optimizer.
    """
    # Suggest optimizer name from a predefined categorical set
    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "AdamW",
            "SGD",
            "Adam",
            "RMSprop",
            "Nadam",
            "Lion",
        ],
    )

    # Suggest learning rate on a logarithmic scale between 1e-5 and 1e-2
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Mapping of optimizer names to their TensorFlow classes
    optimizer_map: Dict[str, Type[tf.keras.optimizers.Optimizer]] = {
        "Adam": optimizers.Adam,
        "AdamW": optimizers.AdamW,
        "SGD": optimizers.SGD,
        "RMSprop": optimizers.RMSprop,
        "Nadam": optimizers.Nadam,
        "Lion": optimizers.Lion,
    }

    # Raise error if selected optimizer is not supported in the current context
    if optimizer_name not in optimizer_map:
        raise ValueError(
            f"Optimizer '{optimizer_name}' is not supported. "
            f"Supported optimizers are: {list(optimizer_map.keys())}."
        )

    # Instantiate and return the selected optimizer with suggested learning rate
    return optimizer_map[optimizer_name](learning_rate=learning_rate)

### 4.4. Callbacks

In [13]:
def get_callbacks(trial: optuna.Trial, checkpoint_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        checkpoint_dir (str): Directory where model weights will be saved.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Construct path for saving weights for this specific trial
    checkpoint_path: str = os.path.join(checkpoint_dir, f"trial_{trial.number}.weights.h5")

    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )

    # Save only the best model weights based on monitored metric
    model_checkpoint = callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=monitor,
        save_best_only=True,  # only save weights if val_loss improves
        save_weights_only=True,  # save only the weights (not full model)
        verbose=0,
    )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = NanLossPrunerCallback(trial)

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, model_checkpoint, nan_pruner_callback, pruning_callback]

### 4.5. Scalers

In [14]:
def get_scaler(
    trial: optuna.Trial,
) -> Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
    """
    Suggests and returns a scikit-learn scaler based on Optuna hyperparameter selection.

    Args:
        trial (optuna.Trial): Optuna trial object used to suggest hyperparameters.

    Returns:
        Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
            Instantiated scaler object from scikit-learn.
    """
    # Suggest a scaler name from the list of supported options
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",  # For normally-distributed data
            "MinMaxScaler_-1_1",  # Normalize to [-1, 1] range
            "MinMaxScaler_0_1",  # Normalize to [0, 1] range
            "RobustScaler",  # For data with outliers
            "QuantileTransformer",  # For non-normal or skewed data
            "PowerTransformer",  # For heavy-tailed or skewed data
        ],
    )

    # Return the appropriate scaler instance based on selection
    if scaler_name == "StandardScaler":
        return StandardScaler()
    elif scaler_name == "RobustScaler":
        return RobustScaler()
    elif scaler_name == "QuantileTransformer":
        return QuantileTransformer(output_distribution="normal")
    elif scaler_name == "PowerTransformer":
        return PowerTransformer(method="yeo-johnson")
    elif scaler_name == "MinMaxScaler_0_1":
        return MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        return MinMaxScaler(feature_range=(-1, 1))

    # Catch invalid or unknown choices
    else:
        raise ValueError(f"Unknown scaler selected: {scaler_name}")

## 5. Layers Builders

### 5.1. CNN

In [15]:
def build_cnn2d(
    trial: optuna.Trial,
    x: layers.Layer,
    num_layers: int = 5,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    max_kernel_size: int = 10,
    min_pool_size_dim1: int = 2,
    max_pool_size_dim1: int = 2,
    min_pool_size_dim2: int = 2,
    max_pool_size_dim2: int = 2,
    use_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn",
) -> layers.Layer:
    """
    Builds a 2D CNN where Optuna picks filters, kernels, and pooling window per layer.

    Args:
        trial: Optuna trial object.
        x: Input Keras tensor.
        num_layers: Number of Conv2D+Pool blocks.
        max_filters: Upper bound on filters.
        min_filters: Lower bound on filters.
        filter_step: Step size for filters.
        max_kernel_size: Max kernel dim for height & width.
        min_pool_size_dim1: Min pooling window height.
        max_pool_size_dim1: Max pooling window height.
        min_pool_size_dim2: Min pooling window width.
        max_pool_size_dim2: Max pooling window width.
        use_batch_norm: If True, trial.prunes BatchNorm on/off.
        use_regularization: If True, trial selects kernel/bias/activity regularizers.
        residual_method: One of {None, "beside", "all"}.
        custom_name: Prefix for naming each layer.

    Returns:
        The output tensor after all blocks.
    """

    # Containers for residual strategies
    beside_residual: Optional[layers.Layer] = None
    all_skip_connections: List[layers.Layer] = []

    for layer_idx in range(num_layers):
        # 0) Sample a single pooling window to use in every block
        pool_dim1 = trial.suggest_int(
            f"{custom_name}_pool_size_dim1_{layer_idx}",
            min_pool_size_dim1,
            max_pool_size_dim1,
        )
        pool_dim2 = trial.suggest_int(
            f"{custom_name}_pool_size_dim2_{layer_idx}",
            min_pool_size_dim2,
            max_pool_size_dim2,
        )
        pool_size: Tuple[int, int] = (pool_dim1, pool_dim2)
        
        # 1) Filters
        num_filters = trial.suggest_int(
            f"{custom_name}_filters_layer_{layer_idx}",
            min_filters,
            max_filters,
            step=filter_step,
        )

        # 2) Kernel dims
        kernel_h = trial.suggest_int(f"{custom_name}_kernel_height_{layer_idx}", 1, max_kernel_size)
        kernel_w = trial.suggest_int(f"{custom_name}_kernel_width_{layer_idx}", 1, max_kernel_size)

        # Activation
        activation_fn = get_activation(trial, f"{custom_name}_activation_layer_{layer_idx}")

        # Regularizers
        kernel_reg = (
            get_regularizer(trial, f"{custom_name}_kernel_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )
        bias_reg = (
            get_regularizer(trial, f"{custom_name}_bias_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_regularizer_layer_{layer_idx}")
            if use_regularization
            else None
        )

        # Conv2D
        x = layers.Conv2D(
            filters=num_filters,
            kernel_size=(kernel_h, kernel_w),
            activation=activation_fn,
            padding="same",
            name=f"{custom_name}_conv2d_{layer_idx}",
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
        )(x)

        # 3) Optional BatchNorm
        if use_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_layer_{layer_idx}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{layer_idx}")(x)

        # 4) Residuals
        if residual_method == "beside":
            if layer_idx == 0:
                beside_residual = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{layer_idx}", [True, False]):
                    prev = beside_residual
                    target_ch = x.shape[-1]
                    if prev.shape[-1] != target_ch:
                        prev = layers.Conv2D(
                            filters=target_ch,
                            kernel_size=(1, 1),
                            padding="same",
                            name=f"{custom_name}_res_align_{layer_idx}",
                        )(prev)
                    x = layers.Add(name=f"{custom_name}_res_add_{layer_idx}")([x, prev])
                    beside_residual = x
                else:
                    beside_residual = x

        elif residual_method == "all":
            if layer_idx == 0:
                all_skip_connections = [x]
            else:
                to_add: List[layers.Layer] = []
                for prev_idx, prev_layer in enumerate(all_skip_connections):
                    if trial.suggest_categorical(
                        f"{custom_name}_use_residual_layer_{layer_idx}_{prev_idx}",
                        [True, False],
                    ):
                        prev = prev_layer
                        target_ch = x.shape[-1]
                        if prev.shape[-1] != target_ch:
                            prev = layers.Conv2D(
                                filters=target_ch,
                                kernel_size=(1, 1),
                                padding="same",
                                name=(f"{custom_name}_skip_res_conv2d_" f"{layer_idx}_{prev_idx}"),
                            )(prev)
                        to_add.append(prev)
                if to_add:
                    x = layers.Add(name=f"{custom_name}_res_all_add_{layer_idx}")([x] + to_add)
                all_skip_connections.append(x)

        # 5) MaxPooling with sampled window
        x = layers.MaxPooling2D(pool_size=pool_size, name=f"{custom_name}_maxpool_{layer_idx}")(x)

    return x

## 6. Objective Function

In [16]:
def objective(
    trial: optuna.Trial,
    X: List[np.ndarray],
    y: List[np.ndarray],
    checkpoint_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        checkpoint_dir (str): Path to store checkpoint files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        use_regularization (bool): If True, adds regularization (e.g., L1/L2) to layers to prevent overfitting.
        residual_method (Optional[str]): tyoe of residual connection to use:
            - "beside": Adds residual connections between consecutive layers.
            - "all": test residual connections between all layers.
            - None: No residual connections are applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    # Each trial gets a different seed to split the data
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ————————————————————————————— Prepare the Data ————————————————————————————— #
    s008_lidar_input = X[0]
    s008_coord_input = X[1]
    s008_y_train = y[0]

    (
        x_lidar_train,
        x_lidar_val,
        x_coord_train,
        x_coord_val,
        y_train,
        y_val,
    ) = train_test_split(
        s008_lidar_input,
        s008_coord_input,
        s008_y_train,
        test_size=0.2,
        random_state=trial.number,
        shuffle=True,
    )

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # —————————————————————————————————— Scaler —————————————————————————————————— #
        # scaler = get_scaler(trial)
        # x_coord_train = scaler.fit_transform(x_coord_train)
        # x_coord_val = scaler.transform(x_coord_val)

        # ——————————————————————————————— LiDAR Input ——————————————————————————————— #
        # Input for LiDAR data (e.g., shape: (20, 200, 10))
        x_lidar_input = layers.Input(shape=(20, 200, 10))

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(x_coord_train.shape[1],))

        # Use z-score normalization
        norm_layer = layers.Normalization(axis=1, name="coord_input_normalization")

        # Computes the mean and variance of the input data
        norm_layer.adapt(x_coord_train)

        # Apply normalization to the input data
        x_coord_norm = norm_layer(x_coord_input)

        # Add spatial dimension
        x_coord_norm = layers.Reshape((1, 1, x_coord_input.shape[1]))(x_coord_norm)

        # Tile across the lidar grid
        # So the coordinates are repeated across the 20x200 grid
        x_coord_norm = layers.Lambda(lambda x: tf.tile(x, [1, 20, 200, 1]))(x_coord_norm)

        # ? If using scaler then uncomment the following line
        # x_coord_input = layers.Reshape((1, 1, x_coord_input.shape[1]))(x_coord_input)
        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels: (batch,20,200,10) + (batch,20,200,2) → (batch,20,200,12)
        combined = layers.Concatenate(axis=-1)([x_lidar_input, x_coord_norm])

        max_layers = trial.suggest_int("cnn_num_layers", 1, 4)

        # Calculate max pool size
        max_pool_dim1 = math.floor(20 ** (1.0 / max_layers))
        max_pool_dim2 = math.floor(200 ** (1.0 / max_layers))

        x = build_cnn2d(
            trial=trial,
            x=combined,
            num_layers=max_layers,
            max_filters=128,
            min_filters=16,
            filter_step=4,
            max_kernel_size=7,
            min_pool_size_dim1=1,
            max_pool_size_dim1=max_pool_dim1,
            min_pool_size_dim2=1,
            max_pool_size_dim2=max_pool_dim2,
            use_batch_norm=True,
            use_regularization=use_regularization,
            residual_method=residual_method,
        )
        # ———————————————————————————— Flatten the Output ———————————————————————————— #
        x = layers.Flatten(name="flatten")(x)

        # ——————————————————————————————— Dense Layers ——————————————————————————————— #
        num_dense_layers = trial.suggest_int("num_dense_layers", 0, 2)
        for i in range(num_dense_layers):
            # Suggest the number of units for each dense layer
            units = trial.suggest_int(f"dense_{i+1}_units", 64, 512, step=64)
            x = layers.Dense(
                units=units,
                activation=get_activation(trial, f"dense_{i+1}_activation"),
                name=f"dense_{i+1}",
            )(x)
            x = layers.Dropout(rate=0.5)(x)

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="softmax")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

        # ———————————————————————————— Vizualize the Model ——————————————————————————— #
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = get_optimizer(trial)
        model.compile(
            optimizer=optimizer,
            loss=losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, checkpoint_dir),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = troo.compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = troo.compute_params_penalized_loss(loss=loss, model=model)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                 Trial Results                                #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present

        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)

        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [s009_lidar_input, s009_coord_input], s009_y, batch_size=batch_size, verbose=0
        )

        trial.set_user_attr("test_accuracy_s009", float(test_acc))

        # ————————————————————————————— Print the results ———————————————————————————— #

        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "=" * 15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {loss:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009):     {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}\n")
        
        params = model.count_params()
        print(f"Number of parameters: {params}")
        print(f"Model size: {params * 4 / (1024 ** 2):.2f} MB")
        print("=" * 15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")

        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"❌ Trial {trial.number} hit OOM (ResourceExhaustedError): {oom_err}")

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(oom_err) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    except Exception as e:
        print(f"An error occurred during the trial execution: {e}")
        traceback.print_exc()

        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(e) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf")  # Return bad loss
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [17]:
# resources_dir = os.path.join(RUN_DIR, "resources")
# os.makedirs(resources_dir, exist_ok=True)
# troo.log_resources(log_dir=resources_dir)

In [18]:
_monitor_proc = troo.launch_kernel_monitor(custom_title="RUN_DIR", script_path="./tensoroo/_monitor_kernel_life.py")

[INFO] Launched monitor in gnome-terminal (PID=26551)


## Main

In [19]:
try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, "optuna_study")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "weights": os.path.join(study_dir, "weights"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    checkpoint_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["weights"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    print(f"Initializing study at '{study_dir}'...")

    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            X=[s008_lidar_input, s008_coord_input],
            y=[s008_y_train],
            checkpoint_dir=checkpoint_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            use_regularization=False,
            residual_method=None,  #! Find your backbone first
            show_summary=True,
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    valid_trials = [
        t for t in study.trials
        if t.value is not None and not (math.isnan(t.value) or math.isinf(t.value))
    ]
    sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_train_acc = trial.user_attrs.get("best_train_accuracy", None)
        trial_val_acc = trial.user_attrs.get("best_val_accuracy", None)
        trial_test_acc = trial.user_attrs.get("test_accuracy_s009", None)

        troo.save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            val_accuracy=trial_val_acc,
            train_accuracy=trial_train_acc,
            test_accuracy=trial_test_acc,
            sampler=study.sampler.__class__.__name__,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    cleanup_paths = [
        (checkpoint_dir, "trial_{trial_id}.weights.h5"),
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    troo.analyze_study(study, fig_dir=fig_dir, table_dir=study_dir)

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(1 for t in study.trials if t.state != optuna.trial.TrialState.COMPLETE)
    troo.notify_training_success(
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject=f"🎉 Training Complete - Failed Trials: {failed_trials}",
    )
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

2025-05-02 09:25:27.381077: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_112', 12 bytes spill stores, 12 bytes spill loads

[I 2025-05-02 09:25:28,941] Trial 9 finished with value: 5.378259658813477 and parameters: {'cnn_num_layers': 2, 'cnn_pool_size_dim1_0': 1, 'cnn_pool_size_dim2_0': 11, 'cnn_filters_layer_0': 52, 'cnn_kernel_height_0': 6, 'cnn_kernel_width_0': 2, 'cnn_activation_layer_0': 'relu', 'cnn_use_batch_norm_layer_0': False, 'cnn_pool_size_dim1_1': 1, 'cnn_pool_size_dim2_1': 9, 'cnn_filters_layer_1': 20, 'cnn_kernel_height_1': 2, 'cnn_kernel_width_1': 3, 'cnn_activation_layer_1': 'tanh', 'cnn_use_batch_norm_layer_1': True, 'num_dense_layers': 2, 'dense_1_units': 128, 'dense_1_activation': 'leaky_relu', 'dense_2_units': 128, 'dense_2_activation': 'tanh', 'optimizer': 'AdamW', 'learning_rate': 0.00010672069780970255, 'batch_size': 256}. Best is trial 8 with value: 2.7



# ——————————————————————— Trial 9 Results ——————————————————————— #

Training loss: 5.378259658813
Training accuracy: 0.0102

Validation loss: 5.378259658813
Validation accuracy: 0.2423

Test loss (s009):     5.438642978668
Test accuracy (s009): 0.1705

Number of parameters: 165949
Model size: 0.63 MB

# ———————————————————————————————————————————————————————————————————————————— #


Number of failed trials: 0



/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/matheus/anaconda3/envs/tf-optuna/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,Parameter,Mean,Std,Min,25%,Median,75%,Max
0,params_batch_size,118.400000,84.057123,32.00000,40.000000,128.000000,128.000000,256.000000
1,params_cnn_filters_layer_0,86.400000,33.741501,32.00000,54.000000,102.000000,115.000000,120.000000
2,params_cnn_filters_layer_1,80.000000,40.623709,20.00000,50.000000,90.000000,114.000000,120.000000
3,params_cnn_filters_layer_2,97.000000,24.953290,64.00000,85.000000,102.000000,114.000000,120.000000
4,params_cnn_filters_layer_3,80.000000,45.254834,48.00000,64.000000,80.000000,96.000000,112.000000
5,params_cnn_kernel_height_0,4.500000,1.840894,1.00000,4.000000,4.000000,5.750000,7.000000
6,params_cnn_kernel_height_1,4.000000,1.927248,2.00000,2.000000,4.000000,5.250000,7.000000
7,params_cnn_kernel_height_2,3.250000,1.707825,1.00000,2.500000,3.500000,4.250000,5.000000
8,params_cnn_kernel_height_3,5.500000,2.121320,4.00000,4.750000,5.500000,6.250000,7.000000
9,params_cnn_kernel_width_0,4.200000,2.097618,1.00000,2.250000,4.500000,6.000000,7.000000


,Parameter,Category,Fraction,Count
0,params_cnn_activation_layer_0,relu,0.500000,5
1,params_cnn_activation_layer_0,leaky_relu,0.200000,2
2,params_cnn_activation_layer_0,elu,0.100000,1
3,params_cnn_activation_layer_0,swish,0.100000,1
4,params_cnn_activation_layer_0,sigmoid,0.100000,1
5,params_cnn_activation_layer_1,sigmoid,0.375000,3
6,params_cnn_activation_layer_1,relu,0.250000,2
7,params_cnn_activation_layer_1,tanh,0.250000,2
8,params_cnn_activation_layer_1,swish,0.125000,1
9,params_cnn_activation_layer_2,tanh,0.500000,2


...

,Parameter,Mean,Std,Min,25%,Median,75%,Max
0,params_batch_size,80.000000,67.882251,32.000000,56.000000,80.000000,104.000000,128.000000
1,params_cnn_filters_layer_0,102.000000,14.142136,92.000000,97.000000,102.000000,107.000000,112.000000
2,params_cnn_filters_layer_1,56.000000,NaN,56.000000,56.000000,56.000000,56.000000,56.000000
3,params_cnn_filters_layer_2,92.000000,NaN,92.000000,92.000000,92.000000,92.000000,92.000000
4,params_cnn_filters_layer_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,params_cnn_kernel_height_0,4.000000,0.000000,4.000000,4.000000,4.000000,4.000000,4.000000
6,params_cnn_kernel_height_1,4.000000,NaN,4.000000,4.000000,4.000000,4.000000,4.000000
7,params_cnn_kernel_height_2,1.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
8,params_cnn_kernel_height_3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,params_cnn_kernel_width_0,4.000000,2.828427,2.000000,3.000000,4.000000,5.000000,6.000000


,Parameter,Category,Fraction,Count
0,params_cnn_activation_layer_0,sigmoid,0.5,1
1,params_cnn_activation_layer_0,elu,0.5,1
2,params_cnn_activation_layer_1,swish,1.0,1
3,params_cnn_activation_layer_2,relu,1.0,1
4,params_cnn_use_batch_norm_layer_1,False,1.0,1
5,params_cnn_use_batch_norm_layer_2,True,1.0,1
6,params_dense_1_activation,leaky_relu,0.5,1
7,params_dense_1_activation,sigmoid,0.5,1
8,params_dense_2_activation,sigmoid,1.0,1
9,params_optimizer,Nadam,0.5,1


,Parameter,Mean,Std,Min,25%,Median,75%,Max
0,params_batch_size,128.000000,0.000000,128.000000,128.000000,128.000000,128.000000,128.000000
1,params_cnn_filters_layer_0,86.000000,48.083261,52.000000,69.000000,86.000000,103.000000,120.000000
2,params_cnn_filters_layer_1,72.000000,NaN,72.000000,72.000000,72.000000,72.000000,72.000000
3,params_cnn_filters_layer_2,64.000000,NaN,64.000000,64.000000,64.000000,64.000000,64.000000
4,params_cnn_filters_layer_3,48.000000,NaN,48.000000,48.000000,48.000000,48.000000,48.000000
5,params_cnn_kernel_height_0,5.500000,2.121320,4.000000,4.750000,5.500000,6.250000,7.000000
6,params_cnn_kernel_height_1,2.000000,NaN,2.000000,2.000000,2.000000,2.000000,2.000000
7,params_cnn_kernel_height_2,5.000000,NaN,5.000000,5.000000,5.000000,5.000000,5.000000
8,params_cnn_kernel_height_3,4.000000,NaN,4.000000,4.000000,4.000000,4.000000,4.000000
9,params_cnn_kernel_width_0,5.000000,1.414214,4.000000,4.500000,5.000000,5.500000,6.000000


,Parameter,Category,Fraction,Count
0,params_cnn_activation_layer_0,relu,1.0,2
1,params_cnn_activation_layer_1,sigmoid,1.0,1
2,params_cnn_activation_layer_2,tanh,1.0,1
3,params_cnn_activation_layer_3,tanh,1.0,1
4,params_cnn_use_batch_norm_layer_1,True,1.0,1
5,params_cnn_use_batch_norm_layer_2,False,1.0,1
6,params_cnn_use_batch_norm_layer_3,True,1.0,1
7,params_dense_1_activation,swish,1.0,1
8,params_dense_2_activation,swish,1.0,1
9,params_optimizer,SGD,0.5,1


[INFO] Email sent successfully.


In [20]:
# Kill the monitor kernel life process
if _monitor_proc is not None and _monitor_proc.poll() is None:
    os.killpg(_monitor_proc.pid, signal.SIGINT)